# 05 — Evaluation, error analysis, and comparison write-up

This notebook doesn't retrain anything — it loads the cross-validated results notebooks 03 and 04
already saved to `data/processed/`, puts them side by side, and does the part that's easy to skip but
is usually the most informative: looking at *specific* images where each model succeeded or failed, not
just the aggregate numbers.

In [ ]:
import ast
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.patch_classifier import PatchClassifier
from src.models.sliding_window import detect

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
plt.rcParams["figure.dpi"] = 110

scenes = pd.read_csv(PROCESSED_DIR / "clean_manifest.csv")
scenes["boxes"] = scenes["boxes"].apply(ast.literal_eval)
scenes["scene_id"] = scenes["scene_id"].astype(str)

sw_results = pd.read_csv(PROCESSED_DIR / "sliding_window_cv_results.csv")
yolo_results = pd.read_csv(PROCESSED_DIR / "yolo_cv_results.csv")
sweep_df = pd.read_csv(PROCESSED_DIR / "sliding_window_threshold_sweep.csv")
SW_THRESHOLD = float(sweep_df.loc[sweep_df["f1"].idxmax(), "threshold"])

sw_results["scene_id"] = sw_results["scene_id"].astype(str)
yolo_results["scene_id"] = yolo_results["scene_id"].astype(str)

## Side-by-side comparison

Both models were cross-validated on the exact same 19 scenes and 5 folds, and scored with the exact same
`match_detections` / `precision_recall` functions (IoU >= 0.3) — so this comparison is apples-to-apples,
not two different papers' self-reported numbers.

In [ ]:
def summarize(df: pd.DataFrame, name: str) -> dict:
    tp, fp, fn = df["tp"].sum(), df["fp"].sum(), df["fn"].sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return {"model": name, "tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall}


summary = pd.DataFrame([
    summarize(sw_results, "Sliding-window CNN (from scratch)"),
    summarize(yolo_results, "YOLO11n (fine-tuned)"),
])
summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = range(len(summary))
ax.bar([i - 0.2 for i in x], summary["precision"], width=0.4, label="precision")
ax.bar([i + 0.2 for i in x], summary["recall"], width=0.4, label="recall")
ax.set_xticks(list(x))
ax.set_xticklabels(summary["model"], rotation=10, ha="right")
ax.legend()
ax.set_title("Cross-validated precision / recall (IoU>=0.3, all 19 scenes)")
plt.tight_layout()
plt.show()

## Per-scene breakdown

Where specifically does each model succeed or fail? This is what points us at *which* images to look at
next, rather than staring at aggregate numbers.

In [ ]:
merged = sw_results.merge(yolo_results, on="scene_id", suffixes=("_sw", "_yolo"))[
    ["scene_id", "tp_sw", "fp_sw", "fn_sw", "tp_yolo", "fp_yolo", "fn_yolo"]
]
merged

## Visual error analysis

Load both trained models and look directly at a few informative scenes: one YOLO gets right, one both
models miss, and the sliding-window model's typical failure mode (swamped by false positives).

In [ ]:
sw_model = PatchClassifier()
sw_model.load_state_dict(torch.load(MODELS_DIR / "sliding_window_classifier.pt"))
sw_model.eval()

yolo_model = YOLO(str(MODELS_DIR / "yolo_detector.pt"))


def show_both(scene_id: str, sw_threshold: float = SW_THRESHOLD, yolo_conf: float = 0.4, max_sw_boxes: int = 30):
    scene = scenes[scenes["scene_id"] == scene_id].iloc[0]
    image = Image.open(scene["image_path"])

    sw_dets = detect(sw_model, image, score_threshold=sw_threshold, nms_iou_threshold=0.2)
    yolo_result = yolo_model.predict(image, conf=yolo_conf, verbose=False)[0]

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    for ax, title in zip(axes, ["Sliding-window CNN", "YOLO11n"]):
        ax.imshow(image)
        for (x1, y1, x2, y2) in scene["boxes"]:
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2))
        ax.set_title(f"{title} — scene {scene_id}", fontsize=10)
        ax.axis("off")

    for det in sorted(sw_dets, key=lambda d: d.score, reverse=True)[:max_sw_boxes]:
        x1, y1, x2, y2 = det.box
        axes[0].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linewidth=1, alpha=0.6))
    if len(sw_dets) > max_sw_boxes:
        axes[0].set_title(axes[0].get_title() + f" ({len(sw_dets)} raw detections, showing top {max_sw_boxes})", fontsize=8)

    for box in yolo_result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        axes[1].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linewidth=2))
        axes[1].text(x1, y1 - 4, f"{box.conf.item():.2f}", color="red", fontsize=8)

    plt.tight_layout()
    plt.show()

### Case 1: a scene YOLO localizes correctly

In [ ]:
yolo_success_scenes = yolo_results[yolo_results["tp"] > 0]["scene_id"].tolist()
print(f"YOLO scenes with a true positive: {yolo_success_scenes}")
if yolo_success_scenes:
    show_both(yolo_success_scenes[0])

### Case 2: a scene both models miss

In [ ]:
both_miss = merged[(merged["tp_sw"] == 0) & (merged["tp_yolo"] == 0) & (merged["fn_sw"] > 0)]["scene_id"].tolist()
print(f"Scenes both models fully miss: {both_miss}")
if both_miss:
    show_both(both_miss[0])

### Case 3: the sliding-window model's typical false-positive flood

In [ ]:
worst_fp_scene = sw_results.loc[sw_results["fp"].idxmax(), "scene_id"]
print(f"Scene with the most sliding-window false positives: {worst_fp_scene}")
show_both(worst_fp_scene)

## Practical comparison: size and speed

In [ ]:
sw_params = sum(p.numel() for p in sw_model.parameters())
yolo_params = sum(p.numel() for p in yolo_model.model.parameters())

practical = pd.DataFrame([
    {"model": "Sliding-window CNN", "parameters": sw_params, "inference": "thousands of batched forward passes/image"},
    {"model": "YOLO11n", "parameters": yolo_params, "inference": "single forward pass/image"},
])
practical

## Write-up: methodology, results, failure cases, what was learned

**Methodology.** Both detectors were trained and cross-validated on the same 19 hand-verified "hidden in
a crowd" scenes (notebook 01 filtered these out of a 65-scene source dataset that turned out to be
mostly unrelated portrait closeups), using the same scene-level 5-fold split (notebook 02) and the same
IoU-matched precision/recall/localization-error metrics (`src/eval/metrics.py`) for evaluation — never
accuracy, and never a framework's own self-reported metric for one model without also computing it for
the other.

**Results.** See the comparison table and plot above. The sliding-window CNN, despite a properly
hand-implemented pipeline (IoU, NMS, multi-scale windows, a real threshold sweep, and a documented
hard-negative-mining attempt), tops out at near-zero precision on full scenes. YOLO11n, fine-tuned from
COCO-pretrained weights on the exact same 19 scenes, does substantially better — quantifying exactly how
much transfer learning and end-to-end detection training buys over an isolated patch classifier at this
data scale.

**Failure cases.** The visual cases above show the sliding-window model's dominant failure mode directly:
it doesn't fail by missing Waldo and staying quiet — it fails by calling *everything* a plausible match,
flooding the image with detections that NMS can't fully clean up because they're spread across scales and
positions with genuinely low mutual IoU. That's a distribution-mismatch problem (training crops are
clean and centered; inference crops are not) compounded by too little data to correct it — confirmed by
hard-negative mining making recall *worse*, not better, when 172 mined negatives overwhelmed ~126 real
positive examples.

**What was learned** (mapped to the project's explicit learning objectives):
- *IoU, NMS, sliding windows*: implemented and used from scratch throughout (`src/eval/box_utils.py`,
  `src/models/sliding_window.py`) — not imported from a detection framework.
- *Why detection metrics differ from accuracy*: quantified directly in notebook 01 (the ~1:2345 window
  imbalance), and it's precisely why a near-zero-precision model can still look deceptively okay under
  the wrong metric.
- *Scene-level splitting*: verified explicitly, twice — once on the raw Roboflow split (notebook 01),
  once on our own k-fold assignment (notebook 02) — rather than assumed.
- *Sliding-window-to-YOLO conceptual bridge*: made concrete by literally building both on the same data
  and evaluation code, rather than asserting YOLO is "a smarter version" without a controlled comparison.
- *Small-object detection challenges in a low-data regime*: this notebook's central result — extreme
  scale, heavy imbalance, and a dataset that shrank from 65 to 19 usable scenes after cleaning, are
  jointly what made the from-scratch approach struggle, and augmentation/hard-negative mining alone
  weren't enough to fully compensate.